In [1]:
from llama_index.core import KnowledgeGraphIndex
from llama_index.core import StorageContext, load_index_from_storage
from llama_index.core.graph_stores import SimpleGraphStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5") 
Settings.embed_model = embed_model
Settings.llm = None
Settings.chunk_size = 2

[nltk_data] Downloading package punkt_tab to c:\Users\M LAKSHMI
[nltk_data]     NARASIMHA\anaconda3\envs\ramayan\Lib\site-
[nltk_data]     packages\llama_index\core\_static/nltk_cache...
[nltk_data]   Package punkt_tab is already up-to-date!
c:\Users\M LAKSHMI NARASIMHA\anaconda3\envs\ramayan\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LLM is explicitly disabled. Using MockLLM.


In [2]:
import json
with open(r'D:\Ramayan\abc.json', 'r') as file:
    data = json.load(file)

In [3]:
data_new = [i["text"] for i in data["paragraphs"]]
data_new

['NeoCity is a major innovation hub where Helios Labs operates under the guidance of Dr. Mira Kwan.',
 'Project Atlas plays a critical role in improving infrastructure across NeoCity.',
 'Helios Labs collaborates with city officials to deploy AI solutions.',
 'Dr. Mira Kwan is a key figure in NeoCity’s AI transformation.',
 'Funding from Orion Foundation accelerates innovation in Helios Labs.',
 'Urban systems in NeoCity rely heavily on data-driven platforms.',
 'City Council ensures ethical deployment of AI technologies.',
 'Machine learning expertise drives the success of Project Atlas.',
 'Scientific collaboration strengthens NeoCity’s innovation ecosystem.',
 'The future of NeoCity depends on responsible AI development.']

In [4]:
parsed_data = [i["triplets"] for i in data["paragraphs"]]
parsed_data_new = []
for item in parsed_data:
    dt = []
    for i in item:
        dt.append([i['subject'], i['predicate'], i['object']])
    parsed_data_new.append(dt)
parsed_data = parsed_data_new
parsed_data

[[['NeoCity', 'is_a', 'Innovation Hub'],
  ['Narasimha', 'is_a', 'batman'],
  ['Helios Labs', 'located_in', 'NeoCity'],
  ['Dr. Mira Kwan', 'works_at', 'Helios Labs'],
  ['Helios Labs', 'develops', 'Project Atlas'],
  ['Project Atlas', 'focuses_on', 'Artificial Intelligence'],
  ['NeoCity', 'governed_by', 'City Council'],
  ['Helios Labs', 'funded_by', 'Orion Foundation'],
  ['Dr. Mira Kwan', 'specializes_in', 'Machine Learning'],
  ['Project Atlas', 'used_for', 'Urban Optimization'],
  ['Orion Foundation', 'supports', 'Scientific Research']],
 [['Narasimha', 'is_a', 'batman'],
  ['Project Atlas', 'improves', 'Infrastructure'],
  ['NeoCity', 'uses', 'Project Atlas'],
  ['Helios Labs', 'develops', 'Project Atlas'],
  ['Dr. Mira Kwan', 'leads', 'Project Atlas'],
  ['Infrastructure', 'part_of', 'Urban Systems'],
  ['NeoCity', 'contains', 'Urban Systems'],
  ['Helios Labs', 'located_in', 'NeoCity'],
  ['Project Atlas', 'focuses_on', 'Artificial Intelligence'],
  ['Orion Foundation', 'funds

In [5]:
from llama_index.core import Document
documents = []
for j, i in enumerate(data_new):       
    document = Document(
    text = i,
    metadata={
        "id": j
    },
    excluded_llm_metadata_keys=["r_id"],
    metadata_seperator="::",
    metadata_template="{key}=>{value}",
    text_template="Content: {content}, Extra Chapter context for reference: {metadata_str}\n",
    )
    documents.append(document)
print(len(documents), len(parsed_data))
print(f"Document Metadata: {documents[-1].metadata}")

10 10
Document Metadata: {'id': 9}


In [6]:
# import json
# with open(r'D:\Ramayan\abc.json', 'r') as file:
#     parsed_data = json.load(file)

# file = 'balakanda_chapter1.json'
# import json
# with open(file) as f:
#     data = json.load(f)

# from llama_index.core import Document
# documents = []
# for i in data[:]:       
#     document = Document(
#     text = i['content'],
#     metadata={
#         "canto": i['chapter'],
#         "canto_summary": i['summary'],
#         "r_id": str(i['chapter']) + "_" + str(i['prompt_number'])
#     },
#     excluded_llm_metadata_keys=["r_id"],
#     metadata_seperator="::",
#     metadata_template="{key}=>{value}",
#     text_template="Content: {content}, Extra Chapter context for reference: {metadata_str}\n",
#     )
#     documents.append(document)
# print(len(documents), len(parsed_data))
# print(f"Document Metadata: {documents[-1].metadata}")

In [7]:
from llama_index.core.node_parser import SentenceSplitter
node_parser = SentenceSplitter()
nodes = node_parser.get_nodes_from_documents(documents)

In [8]:
graph_store = SimpleGraphStore()
storage_context = StorageContext.from_defaults(graph_store=graph_store)

index = KnowledgeGraphIndex([], 
                            storage_context=storage_context, 
                            include_embeddings=True)
from tqdm import tqdm
#triplet: Tuple[str, str, str]
for i in tqdm(range(0, len(nodes))):
    for value in parsed_data[i]:
        index.upsert_triplet_and_node(tuple(value), nodes[i], include_embeddings=True)

C:\conda_tmp\ipykernel_13764\2777388544.py:4: DeprecationWarning: Call to deprecated class KnowledgeGraphIndex. (The KnowledgeGraphIndex class has been deprecated. Please use the new PropertyGraphIndex class instead. If a certain graph store integration is missing in the new class, please open an issue on the GitHub repository or contribute it!) -- Deprecated since version 0.10.53.
  index = KnowledgeGraphIndex([],
100%|██████████| 10/10 [00:07<00:00,  1.32it/s]


In [9]:
index.storage_context.persist(persist_dir='mln_test')

In [10]:
query_engine = index.as_query_engine(
    include_text=True, 
    response_mode="tree_summarize",
    embedding_mode="keyword",
    similarity_top_k=5,
    verbose=True)

you the goat buddy, we in KGTableRetriever now!


In [11]:
from llama_index.core.schema import NodeWithScore, QueryBundle
query = QueryBundle("Narasiha")
response = query_engine.retrieve(query)

you the goat buddy, we in _retrieve of KGTableRetriever now!
Extracted keywords: ['Narasiha']
KG context:
The following are knowledge sequence in max depth 2 in the form of directed graph like:
`subject -[predicate]->, object, <-[predicate_next_hop]-, object_next_hop ...`
('Dr. Mira Kwan', 'contributes_to', 'Urban Systems')
('NeoCity', 'benefits_from', 'Project Atlas')
('NeoCity', 'relies_on', 'Urban Systems')
('NeoCity', 'contains', 'Urban Systems')
('Narasimha', 'is_a', 'batman')


In [12]:
response[0]

NodeWithScore(node=TextNode(id_='1a8df9e5-c9c8-41f2-8095-1424d663681e', embedding=None, metadata={'id': 1}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=['r_id'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='76e650cd-5792-4fd4-a086-32b24be8e7ba', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'id': 1}, hash='a1371ec3c2df1dc974573f84a5f4c59f38e19a8367b7aa8c3cbbe7268277038c')}, text='Project Atlas plays a critical role in improving infrastructure across NeoCity.', mimetype='text/plain', start_char_idx=0, end_char_idx=79, text_template='Content: {content}, Extra Chapter context for reference: {metadata_str}\n', metadata_template='{key}=>{value}', metadata_seperator='::'), score=1000.0)

In [13]:
from llama_index.core.indices.knowledge_graph.retrievers import (
    KGRetrieverMode,
    KGTableRetriever,
)
retriever_mode = KGRetrieverMode.HYBRID

In [47]:
ist = index.get_index_struct()

In [50]:
ist.search_node_by_keyword("batman")

['f0631003-7427-47e1-9290-80b68412500b',
 '1f0a7103-d435-47f6-87bb-3c9a0be3a799',
 '1a8df9e5-c9c8-41f2-8095-1424d663681e',
 'e7e5b293-8951-44a4-9e93-d963f7ed5f17',
 '0a336190-6f21-405e-9907-ffd42f67c114',
 '12f0b864-ed96-4b3b-8de2-12acf6a93b7d',
 '7afba566-df21-433d-af7f-b6b3c9c2642f']

In [16]:
from llama_index.core.indices.query.embedding_utils import get_top_k_embeddings

all_rel_texts = list(ist.embedding_dict.keys())

rel_text_embeddings = [
    ist.embedding_dict[_id] for _id in all_rel_texts
]

In [17]:
from llama_index.core.settings import embed_model_from_settings_or_context
qembd = embed_model_from_settings_or_context(
            Settings, index.service_context
        ).get_text_embedding(
                "ai"
            )
similarities, top_rel_texts = get_top_k_embeddings(
    qembd,
    rel_text_embeddings,
    similarity_top_k=5,
    embedding_ids=all_rel_texts,
)
top_rel_texts

["('AI Solutions', 'based_on', 'Machine Learning')",
 "('AI Technologies', 'based_on', 'Machine Learning')",
 "('Responsible AI', 'based_on', 'Machine Learning')",
 "('Project Atlas', 'part_of', 'AI Research')",
 "('AI Research', 'advances', 'Machine Learning')"]

In [45]:
_graph_store = index.graph_store

In [46]:
_graph_store._data

SimpleGraphStoreData(graph_dict={'NeoCity': [['is_a', 'Innovation Hub'], ['governed_by', 'City Council'], ['uses', 'Project Atlas'], ['contains', 'Urban Systems'], ['benefits_from', 'AI Solutions'], ['invests_in', 'AI Research'], ['hosts', 'Helios Labs'], ['relies_on', 'Urban Systems'], ['governed_by', 'City Council'], ['benefits_from', 'Project Atlas'], ['has', 'Innovation Ecosystem'], ['future_depends_on', 'Responsible AI'], ['supports', 'AI Innovation']], 'Narasimha': [['is_a', 'batman'], ['is_a', 'batman'], ['is_a', 'batman'], ['is_a', 'batman'], ['is_a', 'batman'], ['is_a', 'batman'], ['is_a', 'batman']], 'Helios Labs': [['located_in', 'NeoCity'], ['develops', 'Project Atlas'], ['funded_by', 'Orion Foundation'], ['develops', 'Project Atlas'], ['located_in', 'NeoCity'], ['collaborates_with', 'City Council'], ['deploys', 'AI Solutions'], ['funded_by', 'Orion Foundation'], ['located_in', 'NeoCity'], ['conducts', 'AI Research'], ['develops', 'AI Solutions'], ['located_in', 'NeoCity'],

In [20]:
rel_map = _graph_store.get_rel_map(
                    list({"Ethical AI"}), 2
                )

In [21]:
[
    str(rel_obj)
    for rel_objs in rel_map.values()
    for rel_obj in rel_objs
]

["['Ethical AI', 'applied_in', 'NeoCity']",
 "['NeoCity', 'is_a', 'Innovation Hub']",
 "['NeoCity', 'governed_by', 'City Council']",
 "['NeoCity', 'uses', 'Project Atlas']",
 "['NeoCity', 'contains', 'Urban Systems']",
 "['NeoCity', 'benefits_from', 'AI Solutions']",
 "['NeoCity', 'invests_in', 'AI Research']",
 "['NeoCity', 'hosts', 'Helios Labs']",
 "['NeoCity', 'relies_on', 'Urban Systems']",
 "['NeoCity', 'governed_by', 'City Council']",
 "['NeoCity', 'benefits_from', 'Project Atlas']",
 "['NeoCity', 'has', 'Innovation Ecosystem']",
 "['NeoCity', 'future_depends_on', 'Responsible AI']",
 "['NeoCity', 'supports', 'AI Innovation']"]

In [41]:
index.docstore.add_documents([nodes[0]], allow_update=True)
# from llama_index.core.storage.docstore.simple_docstore import SimpleDocumentStore
# docstore = SimpleDocumentStore()